In [ ]:
"""
AUTOMATIC ENVIRONMENT SETUP — Run this cell on ANY PC.
It will find the right Python, create a .venv, install exact library
versions, register a Jupyter kernel, and restart into it automatically.
"""
import subprocess, sys, os, glob, shutil, json, platform

# ── Configuration ──────────────────────────────────────────────────────
PROJECT_DIR = os.path.dirname(os.path.abspath("__file__"))
VENV_DIR = os.path.join(PROJECT_DIR, ".venv")
KERNEL_NAME = "image-project"
REQUIRED_PYTHON = "3.13"          # exact minor version to match your PC
REQUIRED_PYTHON_TUPLE = (3, 13)

REQUIRED_PACKAGES = {
    "Pillow": "11.2.1",
    "numpy": "2.2.5",
    "opencv-python": "4.11.0.86",
    "gradio": "5.29.0",
    "rembg": "2.0.65",
}

# ── Helpers ────────────────────────────────────────────────────────────
def _run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    return r.returncode == 0, r.stdout.strip(), r.stderr.strip()

def _venv_exe(name):
    if os.name == "nt":
        return os.path.join(VENV_DIR, "Scripts", name)
    return os.path.join(VENV_DIR, "bin", name)

def _python_version(exe):
    """Return (major, minor) for a Python executable, or None."""
    try:
        ok, out, _ = _run([exe, "-c", "import sys; print(sys.version_info.major, sys.version_info.minor)"])
        if ok:
            parts = out.strip().split()
            return (int(parts[0]), int(parts[1]))
    except Exception:
        pass
    return None

def _find_python():
    """
    Find a Python that matches REQUIRED_PYTHON_TUPLE.
    Search order:
      1. Current interpreter
      2. Windows 'py' launcher  (py -3.13)
      3. PATH-based names       (python3.13, python3, python)
      4. Common install dirs on Windows & Linux/Mac
    """
    major, minor = REQUIRED_PYTHON_TUPLE

    # 1) Current interpreter
    if sys.version_info[:2] == (major, minor):
        return sys.executable

    candidates = []

    # 2) Windows py launcher
    if os.name == "nt":
        candidates.append(f"py -{major}.{minor}")

    # 3) PATH-based names
    candidates += [
        f"python{major}.{minor}",
        f"python{major}",
        "python",
    ]

    for c in candidates:
        parts = c.split()          # handle "py -3.13"
        ver = _python_version(parts)
        if ver == (major, minor):
            return parts[0] if len(parts) == 1 else c

    # 4) Common install directories (Windows)
    if os.name == "nt":
        search_roots = [
            os.path.expandvars(r"%LOCALAPPDATA%\Programs\Python"),
            r"C:\Python",
            os.path.expandvars(r"%APPDATA%\Local\Programs\Python"),
        ]
        for root in search_roots:
            pattern = os.path.join(root, f"Python{major}{minor}*", "python.exe")
            for p in glob.glob(pattern):
                if _python_version([p]) == (major, minor):
                    return p
    else:
        # Linux / Mac
        for d in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.pyenv/versions")]:
            pattern = os.path.join(d, f"python{major}.{minor}*")
            for p in sorted(glob.glob(pattern)):
                if _python_version([p]) == (major, minor):
                    return p

    return None

# ── Step 1: Find correct Python ───────────────────────────────────────
print("=" * 60)
print("  AUTOMATIC ENVIRONMENT SETUP")
print("=" * 60)

venv_python = _venv_exe("python.exe" if os.name == "nt" else "python")
venv_pip    = _venv_exe("pip.exe"    if os.name == "nt" else "pip")
already_in_venv = (
    os.path.isfile(venv_python) and
    os.path.normcase(os.path.dirname(sys.executable)) ==
    os.path.normcase(os.path.dirname(venv_python))
)

if already_in_venv:
    # ── Fast path: already running inside the venv ─────────────────
    print(f"\n✅ Already running in .venv  (Python {sys.version.split()[0]})")
    # Quick package check
    ok, freeze, _ = _run([venv_pip, "freeze"])
    installed = {}
    if ok:
        for line in freeze.splitlines():
            if "==" in line:
                n, v = line.split("==", 1)
                installed[n.lower()] = v
    all_ok = True
    for pkg, ver in REQUIRED_PACKAGES.items():
        key = pkg.lower().replace("-", "_")
        key2 = pkg.lower()
        cur = installed.get(key) or installed.get(key2)
        if cur == ver:
            print(f"  ✅ {pkg}=={ver}")
        else:
            print(f"  ❌ {pkg}: expected {ver}, got {cur}")
            all_ok = False
    if all_ok:
        print("\n✅ All packages correct. You can run the cells below.")
    else:
        print("\n⚠ Some packages mismatch — reinstalling...")
        _run([venv_pip, "install"] + [f"{p}=={v}" for p, v in REQUIRED_PACKAGES.items()] + ["--quiet"])
        print("  Done. Re-run this cell to verify.")
    print("=" * 60)

else:
    # ── Full setup path ────────────────────────────────────────────
    print(f"\n[1/6] Searching for Python {REQUIRED_PYTHON}...")
    found_python = _find_python()
    if found_python is None:
        print(f"  ❌ Python {REQUIRED_PYTHON} not found on this system.")
        print(f"     Please install it from https://www.python.org/downloads/")
        print(f"     (Make sure to check 'Add Python to PATH' during install)")
        raise SystemExit(1)
    # Resolve the actual executable (handles "py -3.13" etc.)
    if " " in found_python:
        # It's a command with args like "py -3.13", resolve to real path
        ok, real_path, _ = _run(found_python.split() + ["-c", "import sys; print(sys.executable)"])
        if ok:
            found_python = real_path
    ver = _python_version([found_python])
    print(f"  Found: {found_python}  (Python {ver[0]}.{ver[1]}) ✅")

    # ── Step 2: Create venv ────────────────────────────────────────
    print(f"\n[2/6] Virtual environment (.venv)...", end="")
    if os.path.isfile(venv_python):
        # Check the existing venv uses the correct Python version
        existing_ver = _python_version([venv_python])
        if existing_ver == REQUIRED_PYTHON_TUPLE:
            print(f" exists with correct Python ✅")
        else:
            print(f"\n  ⚠ Existing .venv uses Python {existing_ver}, removing...")
            shutil.rmtree(VENV_DIR, ignore_errors=True)
            ok, _, err = _run([found_python, "-m", "venv", VENV_DIR])
            if not ok:
                print(f"  ❌ Failed: {err}")
                raise SystemExit(1)
            print(f"  Recreated with Python {REQUIRED_PYTHON} ✅")
    else:
        print(" creating...")
        ok, _, err = _run([found_python, "-m", "venv", VENV_DIR])
        if not ok:
            print(f"  ❌ Failed: {err}")
            raise SystemExit(1)
        print(f"  Created ✅")

    # ── Step 3: Upgrade pip ────────────────────────────────────────
    print(f"\n[3/6] Upgrading pip...", end="")
    _run([venv_python, "-m", "pip", "install", "--upgrade", "pip", "--quiet"])
    print(" ✅")

    # ── Step 4: Install packages ───────────────────────────────────
    print(f"\n[4/6] Installing pinned packages...")
    ok, freeze, _ = _run([venv_pip, "freeze"])
    installed = {}
    if ok:
        for line in freeze.splitlines():
            if "==" in line:
                n, v = line.split("==", 1)
                installed[n.lower()] = v

    to_install = []
    for pkg, ver in REQUIRED_PACKAGES.items():
        key = pkg.lower().replace("-", "_")
        key2 = pkg.lower()
        cur = installed.get(key) or installed.get(key2)
        if cur == ver:
            print(f"  ✅ {pkg}=={ver}")
        else:
            status = f"has {cur}" if cur else "missing"
            print(f"  📦 {pkg}=={ver} ({status})")
            to_install.append(f"{pkg}=={ver}")

    if to_install:
        ok, _, err = _run([venv_pip, "install"] + to_install + ["--quiet"])
        if not ok:
            print(f"  ⚠ Warnings: {err[:200]}")
        print(f"  Installed {len(to_install)} package(s) ✅")

    # Also install ipykernel
    _run([venv_pip, "install", "ipykernel", "--quiet"])

    # ── Step 5: Register Jupyter kernel ────────────────────────────
    print(f"\n[5/6] Registering Jupyter kernel '{KERNEL_NAME}'...", end="")
    ok, _, _ = _run([
        venv_python, "-m", "ipykernel", "install",
        "--user", "--name", KERNEL_NAME,
        "--display-name", f"Python ({KERNEL_NAME})"
    ])
    print(" ✅" if ok else " ⚠ (register manually if needed)")

    # ── Step 6: Verify all imports inside the venv ─────────────────
    print(f"\n[6/6] Verifying imports in venv...")
    verify = (
        "import PIL; print('  ✅ Pillow', PIL.__version__);"
        "import numpy; print('  ✅ numpy', numpy.__version__);"
        "import cv2; print('  ✅ opencv-python', cv2.__version__);"
        "import gradio; print('  ✅ gradio', gradio.__version__);"
        "import rembg; print('  ✅ rembg OK');"
        "print('\\nAll packages verified! ✅')"
    )
    ok, out, err = _run([venv_python, "-c", verify])
    print(out if ok else f"  ❌ {err[:300]}")

    # ── Auto-switch: set the kernel spec and restart ───────────────
    print("\n" + "=" * 60)
    print("  ⏳ Switching notebook to the .venv kernel...")
    print("     The kernel will RESTART. This is expected.")
    print("     After restart, run ALL CELLS (Ctrl+Shift+Enter).")
    print("=" * 60)

    # Write a kernel.json that points directly to our venv python
    # so the kernel spec is self-contained
    kernel_dir = os.path.join(
        os.path.expanduser("~"),
        "AppData", "Roaming", "jupyter", "kernels", KERNEL_NAME
    ) if os.name == "nt" else os.path.join(
        os.path.expanduser("~"),
        ".local", "share", "jupyter", "kernels", KERNEL_NAME
    )
    os.makedirs(kernel_dir, exist_ok=True)
    kernel_json = {
        "argv": [os.path.abspath(venv_python), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
        "display_name": f"Python ({KERNEL_NAME})",
        "language": "python",
        "metadata": {"debugger": True}
    }
    with open(os.path.join(kernel_dir, "kernel.json"), "w") as f:
        json.dump(kernel_json, f, indent=2)

    # Auto-restart the kernel into the new environment
    # This uses the Jupyter/IPython kernel restart mechanism
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("\n⚠ Could not auto-restart. Please manually:")
        print(f"  1. Click the kernel selector (top-right in VS Code)")
        print(f"  2. Choose 'Python ({KERNEL_NAME})'")
        print(f"  3. Re-run all cells")

In [ ]:
"""
ENVIRONMENT VALIDATION GATE
If this cell passes, you are running in the correct .venv
with the correct Python and library versions. The main code
cell below will work identically to the original PC.
"""
import sys, os, importlib

REQUIRED_PYTHON = (3, 13)
REQUIRED = {
    "PIL":    ("Pillow",         "11.2.1",    "__version__"),
    "numpy":  ("numpy",          "2.2.5",     "__version__"),
    "cv2":    ("opencv-python",  "4.11.0.86", "__version__"),
    "gradio": ("gradio",         "5.29.0",    "__version__"),
}

errors = []

# Check Python version
if sys.version_info[:2] != REQUIRED_PYTHON:
    errors.append(
        f"Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]} required, "
        f"but running {sys.version_info[0]}.{sys.version_info[1]}. "
        f"Switch to the 'Python (image-project)' kernel."
    )

# Check we are inside the .venv
venv_marker = os.path.join(os.path.dirname(os.path.abspath("__file__")), ".venv")
exe_dir = os.path.normcase(os.path.dirname(sys.executable))
venv_dir = os.path.normcase(
    os.path.join(venv_marker, "Scripts") if os.name == "nt"
    else os.path.join(venv_marker, "bin")
)
if exe_dir != venv_dir:
    errors.append(
        f"Not running from .venv.  Executable: {sys.executable}\n"
        f"  Expected dir: {venv_dir}\n"
        f"  → Select the 'Python (image-project)' kernel and re-run."
    )

# Check each package version
for mod_name, (display, req_ver, attr) in REQUIRED.items():
    try:
        mod = importlib.import_module(mod_name)
        actual = getattr(mod, attr)
        if actual != req_ver:
            errors.append(f"{display}: expected {req_ver}, got {actual}")
        else:
            print(f"  ✅ {display}=={actual}")
    except ImportError:
        errors.append(f"{display}: NOT INSTALLED")

# Check rembg (no simple __version__)
try:
    import rembg
    print(f"  ✅ rembg OK")
except ImportError:
    errors.append("rembg: NOT INSTALLED")

if errors:
    print("\n❌ ENVIRONMENT ERRORS — do NOT proceed:\n")
    for e in errors:
        print(f"  • {e}")
    print("\nFix: run Cell 0 first, then switch to 'Python (image-project)' kernel.")
    raise RuntimeError("Environment mismatch — see errors above.")
else:
    print(f"\n✅ Environment OK — Python {sys.version.split()[0]} in .venv")
    print("   You can now run the main code cell below.")

In [ ]:
# THE BEST OF THE BEST - ambient + freigestellt (IMPROVED)
import os
import numpy as np
from PIL import Image
import gradio as gr
import tkinter as tk
from tkinter import filedialog
import threading
import time
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor
import glob
import itertools
import cv2

# Import rembg for background removal
from rembg import remove

def select_folder():
    result = [None]
    def open_dialog():
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        result[0] = filedialog.askdirectory()
        root.destroy()
    thread = threading.Thread(target=open_dialog)
    thread.start()
    thread.join()
    return result[0] if result[0] else ""

def count_processable_images(folder_path):
    return len(glob.glob(os.path.join(folder_path, "*.png"))) + \
           len(glob.glob(os.path.join(folder_path, "*.jpg"))) + \
           len(glob.glob(os.path.join(folder_path, "*.jpeg"))) + \
           len(glob.glob(os.path.join(folder_path, "*.webp")))

def has_white_background(image_path, margin=40, threshold=0.7):
    """
    Check if an image has a white background by analyzing edge pixels
    """
    try:
        img = Image.open(image_path).convert("RGB")
        img_np = np.array(img)
        
        h, w = img_np.shape[:2]
        
        # Extract edge regions
        top_edge = img_np[:margin, :].reshape(-1, 3)
        bottom_edge = img_np[-margin:, :].reshape(-1, 3)
        left_edge = img_np[:, :margin].reshape(-1, 3)
        right_edge = img_np[:, -margin:].reshape(-1, 3)
        
        # Combine all edge pixels
        all_edges = np.vstack([top_edge, bottom_edge, left_edge, right_edge])
        
        # Calculate how close pixels are to white
        # White is (255, 255, 255)
        white_distances = np.sum(np.abs(all_edges - 255), axis=1)
        
        # Count pixels that are close to white (within a small distance)
        # Lower distance means whiter pixels
        near_white_count = np.sum(white_distances < 30)  # Increased tolerance for "near white"
        total_edge_pixels = len(all_edges)
        
        white_percentage = near_white_count / total_edge_pixels
        
        print(f"White percentage for {image_path}: {white_percentage:.2%}")
        
        return white_percentage > threshold
    except Exception as e:
        print(f"Error checking background for {image_path}: {e}")
        return False

def find_object_bbox_aggressive(image_path, has_white_bg):
    """Find the bounding box of the main object using multiple methods"""
    # Load image
    img = Image.open(image_path).convert("RGBA")
    img_np = np.array(img)
    img_cv = cv2.cvtColor(img_np[:, :, :3], cv2.COLOR_RGBA2BGR)
    
    # Method 1: Using rembg alpha channel
    try:
        removed_bg = remove(img_np)
        alpha = removed_bg[:, :, 3]
        
        # Find non-transparent pixels
        non_transparent = np.where(alpha > 50)  # Lower threshold for better detection
        
        if len(non_transparent[0]) > 0:
            y_min, y_max = non_transparent[0].min(), non_transparent[0].max()
            x_min, x_max = non_transparent[1].min(), non_transparent[1].max()
            
            # Add a margin
            margin = 5
            x_min = max(0, x_min - margin)
            y_min = max(0, y_min - margin)
            x_max = min(img_np.shape[1] - 1, x_max + margin)
            y_max = min(img_np.shape[0] - 1, y_max + margin)
            
            return (x_min, y_min, x_max, y_max)
    except Exception as e:
        print(f"Error with rembg: {e}")
    
    # Method 2: For dark backgrounds, use inverse detection
    if not has_white_bg:
        # For dark backgrounds, detect the main object differently
        gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        
        # Use Otsu's thresholding
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            # Get the largest contour
            largest_contour = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(largest_contour)
            
            # Add margin
            margin = 10
            x_min = max(0, x - margin)
            y_min = max(0, y - margin)
            x_max = min(img_np.shape[1] - 1, x + w + margin)
            y_max = min(img_np.shape[0] - 1, y + h + margin)
            
            return (x_min, y_min, x_max, y_max)
    
    # Method 3: Edge detection for complex cases
    edges = cv2.Canny(img_cv, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        # Get the largest contour
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        
        # Add margin
        margin = 15
        x_min = max(0, x - margin)
        y_min = max(0, y - margin)
        x_max = min(img_np.shape[1] - 1, x + w + margin)
        y_max = min(img_np.shape[0] - 1, y + h + margin)
        
        return (x_min, y_min, x_max, y_max)
    
    # If all else fails, return full image bounds
    return (0, 0, img_np.shape[1] - 1, img_np.shape[0] - 1)

def center_crop_to_square(img, final_size):
    """
    Center crop image to a square of final_size x final_size.
    If image is smaller than final_size in any dimension, it will be scaled up first.
    """
    width, height = img.size
    
    # First, scale the image so that the smaller dimension equals final_size
    if width < height:
        # Width is smaller, scale based on width
        scale_factor = final_size / width
        new_width = final_size
        new_height = int(height * scale_factor)
    else:
        # Height is smaller or equal, scale based on height
        scale_factor = final_size / height
        new_height = final_size
        new_width = int(width * scale_factor)
    
    # Resize the image
    img_resized = img.resize((new_width, new_height), Image.LANCZOS)
    
    # Now center crop to final_size x final_size
    left = (new_width - final_size) // 2
    top = (new_height - final_size) // 2
    right = left + final_size
    bottom = top + final_size
    
    img_cropped = img_resized.crop((left, top, right, bottom))
    
    return img_cropped

def process_single_image(args):
    """Process a single image"""
    input_path, output_path, padding, final_size, has_white_bg = args
    
    try:
        # Load the original image
        img = Image.open(input_path).convert("RGBA")
        
        if has_white_bg:
            # WHITE BACKGROUND: Apply padding and object detection
            # Find the object bounding box with more aggressive detection
            bbox = find_object_bbox_aggressive(input_path, has_white_bg)
            x_min, y_min, x_max, y_max = bbox
            
            # Crop the object with margin
            cropped = img.crop((x_min, y_min, x_max, y_max))
            
            # Calculate dimensions for resizing with padding
            obj_width = x_max - x_min
            obj_height = y_max - y_min
            
            # Apply padding
            available_size = final_size - (2 * padding)
            scale_factor = available_size / max(obj_width, obj_height)
            
            new_width = int(obj_width * scale_factor)
            new_height = int(obj_height * scale_factor)
            
            # Resize the cropped object
            resized = cropped.resize((new_width, new_height), Image.LANCZOS)
            
            # Create final image with padding on white background
            output_img = Image.new("RGBA", (final_size, final_size), (255, 255, 255, 255))
            
            # Calculate position to center the object
            x_offset = (final_size - new_width) // 2
            y_offset = (final_size - new_height) // 2
            
            # Paste the resized object onto the white background
            output_img.paste(resized, (x_offset, y_offset), resized if resized.mode == "RGBA" else None)
            
            print(f"Processed with padding: {input_path}")
            print(f"Object bbox: {bbox}")
            print(f"Resized to: {new_width}x{new_height}")
            print(f"Final size: {final_size}x{final_size}")
        else:
            # NON-WHITE BACKGROUND: Center crop to square
            output_img = center_crop_to_square(img, final_size)
            
            print(f"Processed with center crop: {input_path}")
            print(f"Original size: {img.size}")
            print(f"Final size: {final_size}x{final_size}")
        
        # Save the result (convert to RGB if necessary)
        if output_img.mode == "RGBA":
            # Check if alpha channel is needed
            alpha = np.array(output_img)[:, :, 3]
            if np.all(alpha == 255):
                # No transparency needed, convert to RGB
                output_img = output_img.convert("RGB")
            else:
                # Keep RGBA and save as PNG
                output_path = output_path.replace('.jpg', '.png')
        else:
            output_img = output_img.convert("RGB")
        
        output_img.save(output_path, format="PNG" if output_path.endswith('.png') else "JPEG", quality=95)
        
        return True
    except Exception as e:
        print(f"Error processing {input_path}: {e}")
        return False

def process_images(input_folder, output_folder, padding, final_size, progress=gr.Progress()):
    if not os.path.isdir(input_folder) or not os.path.isdir(output_folder):
        return "Error: Please provide valid folder paths."
    
    # Create output directory
    output_folder_name = f"Output_images_smart_processing"
    full_output_path = os.path.join(output_folder, output_folder_name)
    os.makedirs(full_output_path, exist_ok=True)
    
    # Find all images
    image_files = list(itertools.chain(
        glob.glob(os.path.join(input_folder, "*.png")),
        glob.glob(os.path.join(input_folder, "*.jpg")),
        glob.glob(os.path.join(input_folder, "*.jpeg")),
        glob.glob(os.path.join(input_folder, "*.webp"))
    ))
    
    total_images = len(image_files)
    if total_images == 0:
        return "No images found in the input folder."
    
    start_time = time.time()
    progress(0, desc="Starting...")
    
    # Check background and prepare tasks
    tasks = []
    white_bg_count = 0
    for image in image_files:
        has_white = has_white_background(image)
        if has_white:
            white_bg_count += 1
        
        output_path = os.path.join(full_output_path, os.path.splitext(os.path.basename(image))[0] + ".jpg")
        tasks.append((image, output_path, padding, final_size, has_white))
    
    # Process images
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        results = list(executor.map(process_single_image, tasks))
        
        for i, result in enumerate(results, 1):
            progress(i / total_images, desc=f"Processing {i}/{total_images}")
    
    progress(1.0, desc="Processing complete!")
    total_time = timedelta(seconds=int(time.time() - start_time))
    
    successful = sum(results)
    return f"""Processing completed in {total_time}.
- Processed {successful}/{total_images} images
- {white_bg_count} images had white background (object detection + padding applied)
- {total_images - white_bg_count} images had non-white background (center-cropped to square)
- Output saved to: {full_output_path}"""

def create_gradio_interface():
    with gr.Blocks(title="Smart Image Processor") as app:
        gr.Markdown("# Smart Object Cropping and Resizing Tool")
        gr.Markdown("""**Two processing modes:**
- **White background images:** Detects objects, applies padding, keeps white background
- **Non-white background images:** Center-crops to square (no background changes)""")
        
        with gr.Row():
            with gr.Column():
                input_folder = gr.Textbox(label="Input Folder Path")
                input_btn = gr.Button("Select Input Folder")
                output_folder = gr.Textbox(label="Output Folder Path")
                output_btn = gr.Button("Select Output Folder")
                
                with gr.Row():
                    padding = gr.Slider(minimum=0, maximum=500, value=80, step=10, label="Padding (px) - applied only to white background images")
                    final_size = gr.Slider(minimum=500, maximum=5000, value=1500, step=100, label="Final Image Size (px)")
                
                process_btn = gr.Button("Process Images", variant="primary")
            
            with gr.Column():
                output = gr.Textbox(label="Processing Results", lines=10)
        
        input_btn.click(fn=select_folder, outputs=input_folder)
        output_btn.click(fn=select_folder, outputs=output_folder)
        
        process_btn.click(
            fn=process_images,
            inputs=[input_folder, output_folder, padding, final_size],
            outputs=output
        )
    
    return app

if __name__ == "__main__":
    app = create_gradio_interface()
    app.launch(share=True)